# Evaluación de Tavily como `search_tool` de un agente LangGraph

**Encargo:** el equipo quiere un chatbot de consultas web. Evaluamos Tavily como herramienta de búsqueda: ¿se usa cuando toca?, ¿qué devuelve?, ¿cuánto cuesta en tokens?, ¿es relevante?, ¿se puede mejorar?

**Estado de este notebook:** las celdas de código están listas para ejecutar (hacen falta `ANTHROPIC_API_KEY` y `TAVILY_API_KEY` en `.env`). La sección *Resultados medidos* contiene los datos ya obtenidos de las ejecuciones previas (`agent_haiko_ntb.ipynb`); las tablas de los pasos 2 y 3 se generan al ejecutar. Las conclusiones marcadas como **hipótesis** deben confirmarse con esos datos.

## Resumen (hallazgos con los datos disponibles)
1. En la ejecución con búsqueda, ~**70 %** del input (≈4,8k de 6,95k tokens) es sobrecarga fija — system prompt + schemas de tools — reenviada en **cada** llamada al LLM. Incluso "¿qué es un logaritmo?" cuesta ≈2,4k tokens de input.
2. Los 3 `ToolMessage` pesan ≈**1,99k tokens** (~29 % del input). La métrica "Tokens en Tool Call" del script original mide otra cosa (ver paso 1).
3. Los snippets traen boilerplate de navegación (`label.aria.filter…`) y páginas índice; el `score` no distingue eso (la fuente con menos score aportó el dato de lesiones).
4. El agente afirma "hoy" sin saber la fecha ni tener fechas en los resultados → riesgo de noticias desfasadas.

## 0. Setup

In [4]:
import sys
import json
import pandas as pd
sys.path.insert(0, "../src")
from datetime import date
from dotenv import load_dotenv; load_dotenv()
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain.tools import tool
from websearch_agent import *          # build_agent, make_search_tool, chat, jueces, prompts...
from metrics import *                  # summarize, tool_message_tokens, exact_text_tokens...

MODEL = "claude-haiku-4-5-20251001"
llm = init_chat_model(MODEL, temperature=0)

# Conjunto de consultas: expect_tool = comportamiento esperado del agente
QUERIES = [
    ("Explícame qué es un logaritmo neperiano.",                      False),  # conocimiento estable
    ("¿Quién era el presidente de EE. UU. en enero de 2025?",         False),  # trampa: parece actual, es pre-cutoff
    ("Busca la noticia más importante de hoy del Barça.",             True),   # actualidad
    ("¿Cuál es el precio del Bitcoin ahora mismo?",                   True),   # dato volátil
    ("¿Quién ganó la última Champions League?",                       True),   # depende de la fecha actual
    ("¿Cuánto es 1234 por 5678?",                                     False),  # no necesita web
    ("¿Qué dijo el alcalde de Sant Cugat en el último pleno?",        True),   # fuente de nicho: puede ser infructuosa
    ("Resume las novedades de LangGraph de esta semana.",             True),
]

ModuleNotFoundError: No module named 'pandas'

## 1. Primer paso — ¿el agente decide bien?, acceso a ToolMessages y métricas

### 1.1 Cómo se determinan los tokens de los `ToolMessage` (⚠️ no son los `tool_calls`)

Hay tres cosas distintas que se suelen confundir:

| Concepto | Qué es | Dónde se mide |
|---|---|---|
| **tool_call** | El `AIMessage` donde el modelo *decide* buscar (`tool_use` con `{"query": ...}`) | `usage_metadata` de ese AIMessage: su `output_tokens` ≈ la petición (pocas decenas), su `input_tokens` = todo el contexto enviado en esa llamada |
| **ToolMessage** | El *resultado* de ejecutar la tool (el JSON de Tavily) | **No tiene `usage_metadata`**: lo genera tu código, no el LLM |
| **Tokens del ToolMessage** | Lo que ese texto pesa *cuando se reenvía* al LLM | Se factura dentro del `input_tokens` de la **siguiente** llamada |

El script original (`analyze_token_usage`) suma `input/output` de los `AIMessage` con `tool_calls` y lo llama "Tokens consumidos en Tool Call": eso es el coste de **decidir** buscar, no el de **inyectar el resultado**.

**Método diferencial.** Con llamadas A (decide buscar) y B (la siguiente), el contexto de B es el de A + la respuesta de A + los ToolMessages:

`input_B = input_A + output_A + tokens(ToolMessages) + overhead de formato`

⇒ `tokens(ToolMessages) ≈ input_B − input_A − output_A`

**Ejemplo con los datos reales** (consulta del Barça, 3 resultados): `input_A = 2431`, `output_A = 98`, `input_B = 6954 − 2431 = 4523` ⇒ **4523 − 2431 − 98 = 1994 tokens** (≈665 por resultado). Con varios ToolMessage entre A y B, el total se reparte por nº de caracteres.

**Método exacto (verificación).** Se manda el texto del ToolMessage como mensaje de usuario con `max_tokens=1` y se lee `input_tokens`; se resta el envoltorio de un mensaje mínimo (`exact_text_tokens`). La diferencia con el método diferencial debería ser de decenas de tokens.

**Sobrecarga fija.** `input_A` (2431) ya incluye system prompt + schemas de `bind_tools`. Comparando con una consulta sin búsqueda (2425, 2433) vemos que casi todo es fijo: se paga en *cada* llamada, haya búsqueda o no.

In [ ]:
def run_case(agent, q, stats=None):
    ans, _, new = chat(agent, q)
    s = summarize(new,
                  extra_in=(stats or {}).get("judge_input", 0) - run_case.prev_in,
                  extra_out=(stats or {}).get("judge_output", 0) - run_case.prev_out)
    run_case.prev_in, run_case.prev_out = (stats or {}).get("judge_input", 0), (stats or {}).get("judge_output", 0)
    return ans, new, s
run_case.prev_in = run_case.prev_out = 0

# --- Baseline = lo que hay hoy en agente_haiko.py (2 tools, str(), prompt con cutoff fijo, sin fecha)
raw_tavily = TavilySearch(max_results=3, topic="general")
@tool
def multiply(a: int, b: int) -> int:
    """Multiplica dos numeros enteros a y b."""
    return a * b
baseline = build_agent([multiply, raw_tavily], LEGACY_PROMPT, serializer=legacy_serializer)

rows, store = [], {}
for q, expect in QUERIES:
    ans, new, s = run_case(baseline, q)
    store[q] = (ans, new)
    rows.append({"query": q[:45], "expect_tool": expect, **s, "OK": s["used_tool"] == expect})
df_base = pd.DataFrame(rows)
df_base[["query","expect_tool","used_tool","OK","llm_calls","input","output","toolmsg_tokens","cost_usd"]]

In [ ]:
# Acceso a los ToolMessage y verificación exacta del peso (ejemplo: primera consulta con búsqueda)
q_web = next(q for q, e in QUERIES if e and store[q][1] and any(m.type == "tool" for m in store[q][1]))
_, new = store[q_web]
for m in new:
    if m.type == "tool":
        est = tool_message_tokens(new)[0]["tokens_est"]
        exact = exact_text_tokens(llm, m.content)
        print(f"ToolMessage {m.tool_call_id[:12]}…  chars={len(m.content)}  diferencial≈{est}  exacto≈{exact}")

### 1.2 Fecha y cutoff en el `system_prompt` (en vez de una tool de fecha)
Una tool `get_date` añade su schema (cientos de tokens en **todas** las llamadas) y una llamada extra al LLM cuando se usa. Inyectar `Fecha de hoy` y el cutoff en el system prompt cuesta ~40 tokens, no requiere ida y vuelta y permite comparar fechas de las fuentes con hoy. Implementado en `build_system_prompt()`.

## 2. Segundo paso — entender el `search_tool`

### 2.1 Qué devuelve Tavily (estructura) y qué ve el LLM

In [ ]:
r = raw_tavily.invoke({"query": "Barça noticia importante hoy"})
print("claves de primer nivel:", list(r.keys()))
print("claves por resultado :", list(r["results"][0].keys()))
vacios = [k for k, v in r.items() if v in (None, [], "")]
print("campos vacíos que viajan igualmente al LLM:", vacios)

# Lo que ve el LLM de la tool (lo que bind_tools inyecta en cada input)
schema = raw_tavily.args_schema.model_json_schema()
print(json.dumps(schema, ensure_ascii=False)[:1500], "…")
print("tokens del schema de TavilySearch ≈", exact_text_tokens(llm, json.dumps(schema, ensure_ascii=False)))

# Sobrecarga total medida: input de un 'hola' con y sin tools
from langchain.messages import HumanMessage, SystemMessage
msgs = [SystemMessage(content=LEGACY_PROMPT), HumanMessage(content="hola")]
t_con = count_input_tokens(llm.bind_tools([multiply, raw_tavily]), msgs)
t_sin = count_input_tokens(llm, msgs)
print(f"input con tools={t_con}  sin tools={t_sin}  -> schemas de tools ≈ {t_con - t_sin} tokens por llamada")

**Qué esperar:** la API de Tavily acepta muchos parámetros (`topic`, `search_depth`, `time_range`, `include_domains`, `exclude_domains`, `include_answer`, `include_raw_content`, `include_images`, `chunks_per_source`, …). `langchain-tavily` expone buena parte al LLM como argumentos de la tool, y sus descripciones ocupan la mayor parte de los ≈2k tokens fijos. Además, el LLM rara vez necesita decidir `include_images` o `search_depth`.

### 2.2 Snippets: ¿cómo se generan y son la parte relevante de la página?
`content` es un extracto que genera Tavily (no la página entera); en `search_depth="advanced"` puede ser varios fragmentos separados por `[...]` (visto en los resultados). El algoritmo exacto no es público. Lo comprobamos empíricamente: pedimos también `raw_content` y miramos **dónde** cae el snippet y si es boilerplate.

In [ ]:
import re
raw_full = TavilySearch(max_results=3, topic="general", include_raw_content=True)
r2 = raw_full.invoke({"query": "Barça noticia importante hoy"})
NAV = re.compile(r"(label\.aria|Filtrar|Buscar|Reiniciar|Whatsapp|TikTok|Cookies|Suscríbete)", re.I)
out = []
for x in r2["results"]:
    full = x.get("raw_content") or ""
    chunks = [c.strip() for c in x["content"].split("[...]") if c.strip()]
    found = [full.find(c[:60]) for c in chunks]          # posición de cada fragmento en la página
    out.append({"url": x["url"][:55], "score": round(x["score"], 3), "chars_snippet": len(x["content"]),
                "chars_pagina": len(full), "n_fragmentos": len(chunks),
                "pos_en_pagina": found, "boilerplate_nav": len(NAV.findall(x["content"]))})
pd.DataFrame(out)

**Observación ya disponible** (ejecución previa): el snippet de `fcbarcelona.es` incluye texto de interfaz (`label.aria.filter Filtrar noticias … label.aria.search`) y el de `as.com` es una lista de titulares de una página índice, no un artículo. Son relevantes como *señal*, pobres como *evidencia*.

### 2.3 Score: ¿qué es y sirve para filtrar?
El `score` lo calcula Tavily con un modelo propio de relevancia (no documentado): no es una probabilidad calibrada y solo ordena resultados **dentro de una misma consulta**. Lo contrastamos con un juez LLM (schema Pydantic `relevante / relevancia / explicacion`).

In [ ]:
qs_web = [q for q, e in QUERIES if e]
recs = []
for q in qs_web:
    res = raw_tavily.invoke({"query": q})["results"]
    rep, usage = judge_relevance(llm, q, res)
    if rep is None: continue
    for rr in rep.resultados:
        recs.append({"query": q[:35], "i": rr.index, "score": res[rr.index-1]["score"],
                     "juez_relevancia": rr.relevancia, "juez_relevante": rr.relevante, "porque": rr.explicacion})
df_score = pd.DataFrame(recs)
print("Spearman score vs juez:", df_score[["score","juez_relevancia"]].corr(method="spearman").iloc[0,1].round(2))
print(df_score.groupby(pd.cut(df_score.score, [0,.3,.5,.7,1]), observed=True).juez_relevancia.agg(["count","mean"]))
df_score.sort_values("score")

**Lectura:** si la media del juez es plana entre tramos de score, un umbral fijo descarta tanto lo bueno como lo malo. Dato ya observado: con un umbral de 0,5 se habría perdido la fuente (score 0,465) que contenía el parte médico de Eric García/Koundé que usó la respuesta, y se habría conservado la de navegación (0,587).

### 2.4 ¿El modelo usa las respuestas o alucina?
Descomponemos cada respuesta en afirmaciones y comprobamos cada una contra los ToolMessages (juez con Pydantic `GroundingReport`).

In [ ]:
g_rows = []
for q, e in QUERIES:
    ans, new = store[q]
    ev = "\n".join(m.content for m in new if m.type == "tool")
    if not ev: continue
    rep = groundedness(llm, ans, ev)
    g_rows.append({"query": q[:40], "afirmaciones": len(rep.afirmaciones), "respaldadas": round(rep.tasa, 2),
                   "sin_respaldo": [c.afirmacion[:70] for c in rep.afirmaciones if not c.respaldada]})
pd.DataFrame(g_rows)

**Observación ya disponible (Barça):** los datos concretos (lesiones de Eric y Koundé, citas de Flick, Pedri y Laporta, horario del Clásico) están respaldados por los snippets. Lo **no** respaldado: que sean noticias *de hoy* (los snippets están fechados entre el 10 y el 18 de octubre y no traen `published_date`; el agente ni conocía la fecha) y que sea *la más importante* (inferencia del modelo sobre una lista de titulares).

## 3. Tercer paso — mejorar coste y relevancia

Variantes (todas usan `max_results=3`, mismo modelo):

| | Cambio | Ataca |
|---|---|---|
| **A** | Baseline actual | — |
| **B** | Tool *slim* (el LLM solo ve `query`) + resultado compacto (sin campos vacíos, snippet ≤500 car.) + prompt con fecha/cutoff | sobrecarga fija + peso del ToolMessage |
| **C** | B + `min_score` | resultados de baja puntuación |
| **D** | B + juez LLM (Pydantic) antes de inyectar; si no queda nada → `SIN_RESULTADOS_RELEVANTES` | relevancia + búsquedas infructuosas |
| **E** | B + prompt caching (conversación multi-turno) | reenvío del prefijo |

⚠️ **Trade-offs a medir, no a asumir:** (D) el juez cuesta una llamada extra con ≈ los mismos tokens que quieres ahorrar: solo compensa si descarta mucho o evita reintentos; se contabiliza en `extra_in/out`. (E) Haiku 4.5 exige un prefijo mínimo (≈4096 tokens, verificar en la doc) para cachear: con el prefijo slim *no se activará* salvo en conversaciones largas; las escrituras de caché cuestan 1,25× y las lecturas 0,1×.

In [ ]:
today = date.today()
PROMPT = build_system_prompt(today)
variants = {}
tool_B, st_B = make_search_tool(max_results=3, max_chars=500);                         variants["B"] = (build_agent([tool_B], PROMPT), st_B)
tool_C, st_C = make_search_tool(max_results=3, max_chars=500, min_score=0.5);          variants["C"] = (build_agent([tool_C], PROMPT), st_C)
tool_D, st_D = make_search_tool(max_results=3, max_chars=500, judge_llm=llm);          variants["D"] = (build_agent([tool_D], PROMPT), st_D)

rows = [{**r, "variante": "A"} for r in df_base.to_dict("records")]
answers = {"A": {q: store[q][0] for q, _ in QUERIES}}
for name, (ag, st) in variants.items():
    run_case.prev_in = run_case.prev_out = 0
    answers[name] = {}
    for q, expect in QUERIES:
        ans, new, s = run_case(ag, q, st)
        answers[name][q] = (ans, new)
        ev = "\n".join(m.content for m in new if m.type == "tool")
        g = groundedness(llm, ans, ev).tasa if ev else float("nan")
        rows.append({"variante": name, "query": q[:45], "expect_tool": expect, **s, "OK": s["used_tool"] == expect, "grounded": round(g, 2)})
df_all = pd.DataFrame(rows)
summary = df_all.groupby("variante").agg(input=("input","sum"), output=("output","sum"), toolmsg=("toolmsg_tokens","sum"),
                                         cost_usd=("cost_usd","sum"), acierto_tool=("OK","mean"), grounded=("grounded","mean")).round(4)
summary["ahorro_vs_A_%"] = ((1 - summary.cost_usd / summary.loc["A","cost_usd"]) * 100).round(1)
summary

In [ ]:
# E) Prompt caching en conversación multi-turno (3 turnos con búsqueda) vs sin caché
def convo(cache):
    tool_, _ = make_search_tool(max_results=3, max_chars=500)
    ag = build_agent([tool_], PROMPT, cache=cache)
    hist, tot = [], []
    for q in ["Noticias de hoy sobre inteligencia artificial.", "Amplía la segunda.", "¿Y qué se dice de OpenAI?"]:
        _, hist, new = chat(ag, q, hist)
        tot.append(summarize(new))
    return pd.DataFrame(tot)[["llm_calls","input","cache_read","cache_creation","cost_usd"]].assign(cache=cache)
pd.concat([convo(False), convo(True)])

In [ ]:
# Búsquedas infructuosas: ¿qué se inyecta y qué dice el modelo? (consulta de nicho)
q = QUERIES[6][0]
for name in ["A", "B", "D"]:
    ans, new = (store[q] if name == "A" else answers[name][q])
    tm = [m.content for m in new if m.type == "tool"]
    print(f"--- {name}: ToolMessage={len(tm[0]) if tm else 0} chars")
    print(ans[:300], "\n")

## 4. Resultados medidos (ejecuciones previas, variante A)

| Consulta | ¿Tool? | Input | Output | ToolMessage (dif.) |
|---|---|---|---|---|
| Logaritmo neperiano | no | 2 425 | 495 | — |
| Barça: noticia de hoy | sí (3 resultados) | 6 954 (2 431 + 4 523) | 449 | ≈1 994 |
| Presidente EE. UU. ene-2025 (trampa) | no | 2 433 | 113 | — |

- Comportamiento: las 3 decisiones tool/no-tool fueron correctas.
- Sobrecarga fija ≈2,4k tokens/llamada → en la consulta con búsqueda ≈4,8k de 6,95k (**≈69 %**); ToolMessages ≈ **29 %**.
- Coste de la consulta del Barça a 1 $/5 $ por Mtok (Haiku 4.5, verificar): ≈ 0,0092 $.

*Pega aquí las tablas generadas en los pasos 2 y 3 al ejecutar el notebook.*

## 5. Conclusiones (provisionales → confirmar con los datos de §2–3)
1. **Comportamiento:** el agente respeta bien la regla de cutoff en los casos probados (n=3); ampliar el set (§0) antes de generalizar.
2. **Coste:** la palanca principal no es el ToolMessage sino la **sobrecarga fija de schemas**. Hipótesis: la tool slim (solo `query`) recorta ≈1,5–2k tokens en *todas* las llamadas, también en las que no buscan.
3. **Relevancia:** el `score` es débil como filtro único (ejemplo observado); el juez LLM filtra mejor pero cuesta una llamada. Hipótesis: compensa en consultas infructuosas y cuando la respuesta final evita alucinar.
4. **Fiabilidad temporal:** inyectar fecha en el prompt + `published_date` (usar `topic="news"` o `time_range`) es necesario para responder "hoy" sin desfases.
5. **Caching:** solo útil en conversaciones largas (umbral mínimo de prefijo); medir en §3-E.
6. **Siguiente paso:** ejecutar, rellenar tablas, y si B/D mejoran coste o grounded de forma consistente → preparar el post.